# 27B. Text-Table FN-Masked — Unfrozen-Layer Selection (last `n` layers)

**Text/table counterpart of NB27** (`27.textimage_inbatch_unfronzen_layers`). Same controlled-comparison design, same FN-masking loss (NB28's objective), same GT cadence, same train-or-resume semantics — only the pair type changes:

| Axis | NB27 (text/image) | **NB27B (text/table)** |
|---|---|---|
| **Encoders** | DINOv3 (images) + BGE-M3 (text) | **BGE-M3 only** (both text and table are text-side) |
| **Loss** | FN-masked contrastive | FN-masked contrastive (identical math) |
| **FN pairs** | image-image (`reranked_image_image.pkl`) | **table-table** (`reranked_table_table.pkl`) |
| **Records** | `{bulletin_id, section_title, text, image_path}` | **`{bulletin_id, section_title, text, table_text}`** |
| **Batch size** | 16 (put 32 to accelerate) | **16** (NB28's `TAB_BATCH_SIZE`) |
| **Max length** | 8192 | **2048** (NB28's `TAB_BGE_MAX_LENGTH`) |
| **Projection heads** | `head_text_bge`, `head_image_dinov3` | **`head_text_bge_v2`, `head_table_bge_v2`** |
| **Train size** | 2500 | **2500** |
| **Epochs** | 20 | **20** |
| **Swept axis** | `n_unfrozen_layers` ∈ [0,2,4,6,8] | **same** — [0,2,4,6,8] |
| **GT cadence** | every 2 epochs | **every 2 epochs** |
| **Output dir** | `.../diff_unfrozen_layers/textimage_fn_masked/` | **`.../diff_unfrozen_layers/texttable_fn_masked/`** |

### What `n` means here

For text/table, only **one** encoder is involved (BGE-M3), so `n_unfrozen_layers` refers to **BGE's last `n` encoder layers** — not two encoders as in NB27. `n=0` freezes BGE entirely (heads-only baseline); `n=24` would unfreeze the whole BGE encoder. BGE-M3's `bge_model.encoder.layer` is 24 layers deep, so `n ≤ 24`. `n=8` (last third) is a reasonable upper bound for this sweep.

### `n = 0` — the fully-frozen baseline

- BGE-M3 fully frozen; both projection heads train.
- Optimizer for `n=0` contains only the head param-group.
- Sanity check's gradient test is `n`-aware.

### GT cadence — every 2 epochs

`GT_EVERY_N_EPOCHS = 2` → epochs **2, 4, 6, …, 20** (10 evaluations per run). `per_epoch_gt/` contains exactly those CSVs. Last epoch always evaluated (guarded).

### Controlled-comparison design

All `n` values see the **same 2500 records** (`random.Random(RANDOM_SEED + SIZE)`, sampled from `texttable_records` = 6757), and each run reseeds `np.random.seed(RANDOM_SEED)` before its epoch loop → identical batch order across `n`.

### Requires NB28's text/table pickles

- `vars/texttable_records.pkl` — 6757 records with `{bulletin_id, section_title, text, table_text}`
- `vars/gt_texttable_records.pkl` — 500 GT records with `{gt_idx, text, table_text}`

If missing, run NB28's Section 2 + Section 6 once first.

### Output layout

```
OUTPUT_PARENT = results/runs/section_4/diff_unfrozen_layers/texttable_fn_masked
  img_inbatch_layers_ft.log
  summary_all_n.csv
  summary_per_epoch_all_n.csv
  last_0_layers/  last_2_layers/  last_4_layers/  last_6_layers/  last_8_layers/
    latest_checkpoint.pt
    training_history.csv                    # columns: epoch, texttable_loss
    gt_results.csv
    per_epoch_gt/epoch_02,04,06,...,20_gt_results.csv
    head_text_bge_v2.pt
    head_table_bge_v2.pt
    bge_model_unfrozen.pt
```

### Train-or-resume semantics

| State on disk (per `n`) | Behaviour |
|---|---|
| `last_{n}_layers/gt_results.csv` **exists** | Skip; return existing summary |
| `last_{n}_layers/latest_checkpoint.pt` **exists**, epoch < 20 | Resume |
| `last_{n}_layers/latest_checkpoint.pt` **exists**, epoch ≥ 20 | Finalize only |
| `last_{n}_layers/latest_checkpoint.pt` **missing** | **Train from scratch**, epoch 0 → 20 |

In [ ]:
# Resolve paths from the repository root so notebooks work regardless of the kernel CWD.
from pathlib import Path
import os

def _find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "manifests" / "corpus_manifest.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the PhytoPairs repository root (expected data/manifests/corpus_manifest.csv)."
    )

PROJECT_ROOT = _find_project_root()
BSV_ROOT = PROJECT_ROOT / "3.Dataset Construction"
FINETUNING_ROOT = PROJECT_ROOT / "4.Finetuning"
CASE_STUDY_ROOT = PROJECT_ROOT / "5.BBCH case study"
RESULTS_RUNS_ROOT = PROJECT_ROOT / "results" / "runs" / "section_4"
DEFAULT_HF_CACHE = os.environ.get(
    "HF_HOME", str(Path.home() / ".cache" / "huggingface")
)


## Section 0 — Installations

Same pin set as NB27 / NB28.

In [1]:
# !pip install openpyxl
# !pip install pandas --upgrade
# !pip install accelerate
# !pip install transformers==4.57.6
# !pip install --upgrade huggingface_hub

### No HF login needed — BGE-M3 is not gated (unlike DINOv3).

## Section 1 — Setup, logging, config

Logging, HF cache, and pickle helpers identical to NB27 / NB28.

In [1]:
import json
import os
import re
import sys
import time
import math
import random
import pickle
import logging
import gc
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

import transformers

In [20]:
transformers.logging.set_verbosity_error()

OUTPUT_PARENT = str(RESULTS_RUNS_ROOT / 'layer_sweeps' / 'texttable_fn_masked')
os.makedirs(OUTPUT_PARENT, exist_ok=True)

LOG_FORMAT = '%(asctime)s : %(levelname)s : %(message)s'
LOG_FILE   = os.path.join(OUTPUT_PARENT, "img_inbatch_layers_ft.log")

logging.basicConfig(filename=LOG_FILE, level=logging.INFO, format=LOG_FORMAT)
logger = logging.getLogger("img_inbatch_ft")
if not any(type(h) is logging.StreamHandler for h in logger.handlers):
    _sh = logging.StreamHandler(sys.stdout)
    _sh.setFormatter(logging.Formatter(LOG_FORMAT))
    logger.addHandler(_sh)

for i in range(torch.cuda.device_count()):
   logger.info(torch.cuda.get_device_properties(i).name)
    
device = "cuda:0" if torch.cuda.is_available() else "cpu"
logger.info("Device: %s", device)
if device == "cuda:0":
    logger.info("GPU: %s", torch.cuda.get_device_name(0))

VARS_DIR = str(FINETUNING_ROOT / "vars")
os.makedirs(VARS_DIR, exist_ok=True)

def pkl_vars(varname, filename):
    with open(filename, 'wb') as file:
        pickle.dump(varname, file)

def reload_vars(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)

2026-10-22 11:02:01,008 : INFO : Quadro RTX 8000
2026-10-22 11:02:01,011 : INFO : Quadro RTX 8000
2026-10-22 11:02:01,013 : INFO : Device: cuda:0
2026-10-22 11:02:01,014 : INFO : GPU: Quadro RTX 8000


### Config

- **`SIZE = 2500`** — fixed training-set size for every `n`.
- **`TARGET_EPOCHS = 20`**.
- **`GT_EVERY_N_EPOCHS = 2`** — GT at epochs 2, 4, …, 20.
- **`N_LAYERS_TO_TRY = [0, 2, 4, 6, 8]`** — **`6` included**: you train your own FN-masked `n=6` for text/table; NB28's `texttable` reference is **not** loaded by default (set `NB28_TT_N6_REFERENCE_GT_PATH` in Section 10 only if you want a redundant second row for cross-validation).
- **`TAB_BATCH_SIZE = 16`** — NB28's texttable value.
- **`TAB_BGE_MAX_LENGTH = 2048`** — NB28's texttable value.
- **`GT_EVAL_CHUNK_SIZE = 16`** — chunked GT eval (both text and table sides).
- **`_MODEL_FILE_RENAMES`** — trimmed to `bge_model → bge_model_unfrozen`; head keys pass through unchanged.

In [3]:
# --- Shared ---
HF_CACHE          = DEFAULT_HF_CACHE
RANDOM_SEED       = 42
SIZE              = 2500
TARGET_EPOCHS     = 20
GT_EVERY_N_EPOCHS = 2
N_LAYERS_TO_TRY   = [0, 2, 4, 6, 8]     # 0 = fully frozen baseline; 

# Optional NB28 texttable n=6 reference (leave None to disable -- you train 6 yourself).
# Set to the path below if you want a redundant second n=6 row for cross-validation:
NB28_TT_N6_REFERENCE_GT_PATH = None
# NB28_TT_N6_REFERENCE_GT_PATH = (
#     str(RESULTS_RUNS_ROOT / 'size_sweeps')
#     "Texttable_Smaller_train_sizes_runs/size_2500/gt_results.csv"
# )

# --- Chunking ---
GT_EVAL_CHUNK_SIZE = 32 #was 16, too slow

LOG_EVERY_N_STEPS        = 50
CHECKPOINT_EVERY_N_STEPS = 100
MAX_RUNTIME_HOURS_PER_SIZE = 10.0

K_VALUES          = [1, 3, 5, 10]
HEAD_LEARNING_RATE = 1e-4
BGE_ENCODER_LR    = 1e-5
TEMPERATURE       = 0.07
SHARED_DIM        = 512
TAB_BATCH_SIZE    = 32 #was 16, too slow
TAB_BGE_MAX_LENGTH = 1024 # covers more than 90% of the tables <= p50: 209.0 p90: 860.2000000000016 p99: 2983.3599999999924 max: 7412
BGE_MODEL_NAME    = "BAAI/bge-m3"
FN_SIMILARITY_THRESHOLD = 0.8

_MODEL_FILE_RENAMES = {
    "bge_model": "bge_model_unfrozen",
}

# --- INPUT paths ---
FN_METADATA_PATH   = str(FINETUNING_ROOT / 'Shared_data' / 'df_combined_metadata.pkl')
FN_TABLE_TABLE_PATH = str(FINETUNING_ROOT / 'Shared_data' / 'reranked_table_table.pkl')

logger.info("Config loaded. OUTPUT_PARENT=%s", OUTPUT_PARENT)
logger.info("SIZE=%d, TARGET_EPOCHS=%d, GT_EVERY_N_EPOCHS=%d, N_LAYERS_TO_TRY=%s (0=frozen baseline)",
            SIZE, TARGET_EPOCHS, GT_EVERY_N_EPOCHS, N_LAYERS_TO_TRY)
logger.info("Texttable constants: TAB_BATCH_SIZE=%d, TAB_BGE_MAX_LENGTH=%d, GT_EVAL_CHUNK_SIZE=%d",
            TAB_BATCH_SIZE, TAB_BGE_MAX_LENGTH, GT_EVAL_CHUNK_SIZE)
logger.info("NB28 texttable n=6 reference path: %s", NB28_TT_N6_REFERENCE_GT_PATH or "(disabled)")

2026-10-22 10:32:03,449 : INFO : Config loaded. OUTPUT_PARENT=/projects/agrifutur/sji/cross_encoder_ft/diff_unfrozen_layers/texttable_fn_masked
2026-10-22 10:32:03,451 : INFO : SIZE=2500, TARGET_EPOCHS=20, GT_EVERY_N_EPOCHS=2, N_LAYERS_TO_TRY=[0, 2, 4, 6, 8] (0=frozen baseline)
2026-10-22 10:32:03,453 : INFO : Texttable constants: TAB_BATCH_SIZE=32, TAB_BGE_MAX_LENGTH=1024, GT_EVAL_CHUNK_SIZE=32
2026-10-22 10:32:03,454 : INFO : NB28 texttable n=6 reference path: (disabled)


## Section 2 — Load data (reuse NB28's text/table pickles)

- `vars/texttable_records.pkl` — 6757 records, schema `{orig_idx, bulletin_id, section_title, text, table_text}`
- `vars/gt_texttable_records.pkl` — 500 GT records, schema `{gt_idx, text, table_text}`

If missing, run NB28's Section 2 + Section 6 first.

In [4]:
REQUIRED_PICKLES = {
    "texttable_records":    os.path.join(VARS_DIR, "texttable_records.pkl"),
    "gt_texttable_records": os.path.join(VARS_DIR, "gt_texttable_records.pkl"),
}
missing = [p for p in REQUIRED_PICKLES.values() if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        "Missing NB28 text/table pickles:\n  " + "\n  ".join(missing) +
        "\nRun NB28's Section 2 and Section 6 once to build them, then re-run this cell."
    )

texttable_records    = reload_vars(REQUIRED_PICKLES["texttable_records"])
gt_texttable_records = reload_vars(REQUIRED_PICKLES["gt_texttable_records"])

logger.info("Loaded NB28 text/table pickles: %d training records, %d GT records",
            len(texttable_records), len(gt_texttable_records))
assert len(texttable_records) == 6757, f"expected 6757 records, got {len(texttable_records)}"
assert len(texttable_records) > SIZE, f"pool ({len(texttable_records)}) must exceed SIZE ({SIZE})"
assert len(gt_texttable_records) > 0

_sample = texttable_records[0]
for _k in ("bulletin_id", "section_title", "text", "table_text"):
    assert _k in _sample, f"missing key {_k!r} in texttable_records -- wrong pickle?"
logger.info("Training record schema OK (has bulletin_id/section_title/text/table_text)")

2026-10-22 10:32:03,534 : INFO : Loaded NB28 text/table pickles: 6757 training records, 500 GT records
2026-10-22 10:32:03,536 : INFO : Training record schema OK (has bulletin_id/section_title/text/table_text)


In [5]:
tok = AutoTokenizer.from_pretrained("BAAI/bge-m3", cache_dir=DEFAULT_HF_CACHE)
lens = [len(tok(t, add_special_tokens=False)["input_ids"])
        for t in [r["table_text"] for r in texttable_records[:]]]
print("p50:", np.percentile(lens, 50), "p90:", np.percentile(lens, 90),
      "p99:", np.percentile(lens, 99), "max:", max(lens))

p50: 209.0 p90: 860.2000000000016 p99: 2983.3599999999924 max: 7412


## Section 3 — False-negative mask (table-table, verbatim from NB28)

Precomputed table-table FN pairs at `colbert ≥ FN_SIMILARITY_THRESHOLD` are masked out of the in-batch contrastive loss. Keys are `(bulletin_id, section_title, text)`.

In [6]:
def build_key(bulletin_id, section_title, text):
    return (bulletin_id, section_title, text)

def build_fn_mask_set(pkl_path, threshold=FN_SIMILARITY_THRESHOLD):
    df = pd.read_pickle(pkl_path)
    hi = df[df["colbert"] >= threshold]
    pairs = set()
    for _, row in hi.iterrows():
        pairs.add(frozenset({int(row["global_i"]), int(row["global_j"])}))
    return pairs

fn_metadata = pd.read_pickle(FN_METADATA_PATH)

fn_table_table = build_fn_mask_set(FN_TABLE_TABLE_PATH)
key_to_globals_table = defaultdict(list)
for _, row in fn_metadata[fn_metadata["origin"] == "text_table"].iterrows():
    k = build_key(row["bulletin_id"], row["section_title"], row["text"])
    key_to_globals_table[k].append(row["_global"])
logger.info("Table-table: %d flagged pairs, %d unique keys", len(fn_table_table), len(key_to_globals_table))

def is_masked_table_table(record_i, record_j, threshold_set=fn_table_table):
    key_i = build_key(record_i["bulletin_id"], record_i["section_title"], record_i["text"])
    key_j = build_key(record_j["bulletin_id"], record_j["section_title"], record_j["text"])
    for gi in key_to_globals_table.get(key_i, []):
        for gj in key_to_globals_table.get(key_j, []):
            if frozenset({gi, gj}) in threshold_set:
                return True
    return False

logger.info("is_masked_table_table() ready.")

2026-10-22 10:32:10,511 : INFO : Table-table: 55459 flagged pairs, 6561 unique keys
2026-10-22 10:32:10,514 : INFO : is_masked_table_table() ready.


## Section 4 — Model builder (parameterized by `n_unfrozen_layers`)

Same architecture as NB28's texttable branch. Only change: `n_unfrozen_layers` selects how many of the **last** BGE-M3 encoder layers to unfreeze.

**`n = 0`:** BGE fully frozen; heads train. Optimizer omits the empty BGE param group.

Guards: `n < 0` → asserted away; `n > 24` → warn and unfreeze all.

In [7]:
def build_models_texttable(n_unfrozen_layers):
    assert n_unfrozen_layers >= 0, f"n_unfrozen_layers must be >= 0, got {n_unfrozen_layers}"

    bge_tokenizer = AutoTokenizer.from_pretrained(BGE_MODEL_NAME, cache_dir=HF_CACHE)
    bge_model = AutoModel.from_pretrained(BGE_MODEL_NAME, cache_dir=HF_CACHE).to(device)
    for p in bge_model.parameters():
        p.requires_grad = False
    n_bge = len(bge_model.encoder.layer)
    if n_unfrozen_layers > n_bge:
        logger.warning("n=%d > BGE-M3 layers (%d); unfreezing all", n_unfrozen_layers, n_bge)
    if n_unfrozen_layers > 0:
        for layer in bge_model.encoder.layer[-n_unfrozen_layers:]:
            for p in layer.parameters():
                p.requires_grad = True
    BGE_DIM = bge_model.config.hidden_size

    def encode_bge_text(texts):
        inputs = bge_tokenizer(texts, return_tensors="pt", padding=True, truncation=True,
                                 max_length=TAB_BGE_MAX_LENGTH).to(device)
        out = bge_model(**inputs)
        return out.last_hidden_state[:, 0, :]

    class ProjectionHead(nn.Module):
        def __init__(self, in_dim, out_dim=SHARED_DIM):
            super().__init__()
            self.proj = nn.Linear(in_dim, out_dim)
        def forward(self, x):
            return F.normalize(self.proj(x), p=2, dim=-1)

    head_text_bge_v2  = ProjectionHead(BGE_DIM).to(device)
    head_table_bge_v2 = ProjectionHead(BGE_DIM).to(device)

    # --- Optimizer: only include encoder group if it has trainable params ---
    head_params   = list(head_text_bge_v2.parameters()) + list(head_table_bge_v2.parameters())
    bge_trainable = [p for p in bge_model.parameters() if p.requires_grad]

    param_groups = [{"params": head_params, "lr": HEAD_LEARNING_RATE}]
    if bge_trainable:
        param_groups.append({"params": bge_trainable, "lr": BGE_ENCODER_LR})
    optimizer = torch.optim.AdamW(param_groups)

    logger.info("build_models_texttable(n=%d): BGE trainable params=%d, optimizer groups=%d%s",
                n_unfrozen_layers, len(bge_trainable), len(param_groups),
                " (frozen baseline: heads only)" if n_unfrozen_layers == 0 else "")

    return {
        "bge_model": bge_model, "encode_bge_text": encode_bge_text,
        "head_text_bge_v2": head_text_bge_v2, "head_table_bge_v2": head_table_bge_v2,
        "optimizer": optimizer,
    }

logger.info("build_models_texttable(n) ready.")

2026-10-22 10:32:10,530 : INFO : build_models_texttable(n) ready.


## Section 5 — FN-masked contrastive loss (text/table, verbatim from NB28)

`masked_symmetric_infonce` + `compute_texttable_loss` — same math as NB28's texttable branch. Both text and table go through the same BGE encoder with different projection heads.

In [8]:
def make_batches(records, batch_size, shuffle=True):
    idx = np.arange(len(records))
    if shuffle:
        np.random.shuffle(idx)
    for start in range(0, len(idx), batch_size):
        batch_idx = idx[start:start + batch_size]
        yield [records[i] for i in batch_idx]


def masked_symmetric_infonce(sim_matrix, mask_matrix, temperature=TEMPERATURE):
    B = sim_matrix.shape[0]
    logits = sim_matrix / temperature
    neg_inf = torch.finfo(logits.dtype).min
    mask_matrix = mask_matrix.clone()
    mask_matrix.fill_diagonal_(False)
    logits = logits.masked_fill(mask_matrix, neg_inf)
    targets = torch.arange(B, device=logits.device)
    loss_i2j = F.cross_entropy(logits, targets)
    loss_j2i = F.cross_entropy(logits.T, targets)
    return (loss_i2j + loss_j2i) / 2


def compute_texttable_loss(batch, models):
    texts  = [r["text"] for r in batch]
    tables = [r["table_text"] for r in batch]
    B = len(batch)
    text_emb_raw  = models["encode_bge_text"](texts)
    table_emb_raw = models["encode_bge_text"](tables)
    text_emb  = models["head_text_bge_v2"](text_emb_raw)
    table_emb = models["head_table_bge_v2"](table_emb_raw)
    sim = text_emb @ table_emb.T
    mask = torch.zeros((B, B), dtype=torch.bool, device=device)
    for i in range(B):
        for j in range(B):
            if i != j and is_masked_table_table(batch[i], batch[j]):
                mask[i, j] = True
    return masked_symmetric_infonce(sim, mask)

logger.info("compute_texttable_loss() (FN-masked) ready.")

2026-10-22 10:32:10,586 : INFO : compute_texttable_loss() (FN-masked) ready.


## Section 6 — GT metrics + evaluation function (aligned with NB28)

Both text and table GT embeddings are chunked (`GT_EVAL_CHUNK_SIZE = 16`), matching NB28's texttable branch.

In [9]:
def compute_retrieval_metrics(sim_matrix, k_values=K_VALUES):
    N = sim_matrix.shape[0]
    true_scores = sim_matrix.diag().unsqueeze(1)
    ranks = (sim_matrix >= true_scores).sum(dim=1)
    results = {}
    for k in k_values:
        results[f"R@{k}"] = (ranks <= k).float().mean().item()
    results["MRR"] = (1.0 / ranks.float()).mean().item()
    results["N"] = N
    return results


def encode_bge_text_chunked_eval(texts, models, chunk_size=GT_EVAL_CHUNK_SIZE):
    if len(texts) <= chunk_size:
        return models["encode_bge_text"](texts)
    chunks = []
    for i in range(0, len(texts), chunk_size):
        chunks.append(models["encode_bge_text"](texts[i:i + chunk_size]))
    return torch.cat(chunks, dim=0)


def evaluate_texttable(models):
    models["head_text_bge_v2"].eval(); models["head_table_bge_v2"].eval(); models["bge_model"].eval()
    with torch.no_grad():
        gt_text_emb  = models["head_text_bge_v2"](
            encode_bge_text_chunked_eval([r["text"] for r in gt_texttable_records], models)
        )
        gt_table_emb = models["head_table_bge_v2"](
            encode_bge_text_chunked_eval([r["table_text"] for r in gt_texttable_records], models)
        )
    sim = gt_text_emb @ gt_table_emb.T
    summary = pd.DataFrame([
        {"direction": "text -> table", **compute_retrieval_metrics(sim)},
        {"direction": "table -> text", **compute_retrieval_metrics(sim.T)},
    ])[["direction", "N", "R@1", "R@3", "R@5", "R@10", "MRR"]]
    models["head_text_bge_v2"].train(); models["head_table_bge_v2"].train(); models["bge_model"].train()
    return summary

logger.info("evaluate_texttable() ready.")

2026-10-22 10:32:10,645 : INFO : evaluate_texttable() ready.


## Section 7 — Sanity check (run BEFORE the sweep)

Loss + backward on `n = N_LAYERS_TO_TRY[0]` (= `n=0`). Gradient check is `n`-aware.

Batch size is 16 → naive ln(16) ≈ 2.77 for random init.

In [10]:
_sanity_n = N_LAYERS_TO_TRY[0]
_sanity_models = build_models_texttable(_sanity_n)
_sanity_batch = texttable_records[:TAB_BATCH_SIZE]
_loss_check = compute_texttable_loss(_sanity_batch, _sanity_models)

logger.info("Sanity (n=%d): loss=%.4f (naive ln(%d)=%.4f)",
            _sanity_n, _loss_check.item(), TAB_BATCH_SIZE, math.log(TAB_BATCH_SIZE))
if not math.isfinite(_loss_check.item()):
    raise RuntimeError("Sanity loss is NaN/Inf -- stop and debug before running the sweep")

_loss_check.backward()

_grad_bge = any(p.grad is not None and p.grad.abs().sum().item() > 0
                for p in _sanity_models["bge_model"].parameters() if p.requires_grad)

if _sanity_n == 0:
    logger.info("Sanity (n=0): BGE FROZEN -- absence of encoder gradients is expected.")
    logger.info("  BGE trainable params receive grad: %s (expected False)", _grad_bge)
    _grad_heads = any(p.grad is not None and p.grad.abs().sum().item() > 0
                      for p in _sanity_models["head_text_bge_v2"].parameters()) or \
                  any(p.grad is not None and p.grad.abs().sum().item() > 0
                      for p in _sanity_models["head_table_bge_v2"].parameters())
    logger.info("  projection heads receive grad: %s (expected True)", _grad_heads)
    if not _grad_heads:
        raise RuntimeError("Sanity failed at n=0: no gradients reaching projection heads either")
else:
    logger.info("Gradients reaching unfrozen BGE layers: %s", _grad_bge)
    if not _grad_bge:
        raise RuntimeError(
            f"Sanity failed at n={_sanity_n}: BGE should be receiving gradients. Check optimizer param groups."
        )

del _sanity_models, _loss_check, _sanity_batch
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache(); torch.cuda.synchronize()
logger.info("Allocated after sanity cleanup: %.2f GB", torch.cuda.memory_allocated() / 1e9)

2026-10-22 10:32:14,681 : INFO : build_models_texttable(n=0): BGE trainable params=0, optimizer groups=1 (frozen baseline: heads only)
2026-10-22 10:32:18,470 : INFO : Sanity (n=0): loss=3.5060 (naive ln(32)=3.4657)
2026-10-22 10:32:18,534 : INFO : Sanity (n=0): BGE FROZEN -- absence of encoder gradients is expected.
2026-10-22 10:32:18,536 : INFO :   BGE trainable params receive grad: False (expected False)
2026-10-22 10:32:18,547 : INFO :   projection heads receive grad: True (expected True)
2026-10-22 10:32:19,016 : INFO : Allocated after sanity cleanup: 0.02 GB


## Section 8 — Train-or-resume for one `n` (GT every 2 epochs)

Same logic as NB27's Section 8 — `size_dir` created on fresh start, `per_epoch_gt/` written only when `(epoch+1) % GT_EVERY_N_EPOCHS == 0` or on the last epoch, controlled batch order via `np.random.seed(RANDOM_SEED)`.

In [11]:
def train_or_resume_for_n_layers(n_unfrozen_layers, output_parent, size, records_pool, batch_size,
                                  build_models_fn, loss_fn, eval_fn, state_dict_keys, loss_key,
                                  target_epochs=TARGET_EPOCHS, gt_every_n_epochs=GT_EVERY_N_EPOCHS):
    size_dir = os.path.join(output_parent, f"last_{n_unfrozen_layers}_layers")
    os.makedirs(size_dir, exist_ok=True)
    ckpt_path = os.path.join(size_dir, "latest_checkpoint.pt")
    gt_results_path = os.path.join(size_dir, "gt_results.csv")
    per_epoch_gt_dir = os.path.join(size_dir, "per_epoch_gt")

    label = f"L{n_unfrozen_layers}"

    if os.path.exists(gt_results_path):
        logger.info("[%s] Already completed (%s exists) -- skipping.", label, gt_results_path)
        try:
            return pd.read_csv(gt_results_path)
        except Exception as e:
            logger.warning("[%s] Failed to read existing %s: %s", label, gt_results_path, e)
            return None

    os.makedirs(per_epoch_gt_dir, exist_ok=True)

    rng = random.Random(RANDOM_SEED + size)
    size_records = list(records_pool) if size >= len(records_pool) else rng.sample(records_pool, size)
    logger.info("[%s] Records: %d (fixed subset, seed=%d)", label, len(size_records), RANDOM_SEED + size)

    models = build_models_fn(n_unfrozen_layers)

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        start_epoch = ckpt["epoch"]
        for key in state_dict_keys:
            models[key].load_state_dict(ckpt[key])
        models["optimizer"].load_state_dict(ckpt["optimizer"])
        global_step = ckpt["global_step"]
        history = ckpt.get("history", {"epoch": []})
        if start_epoch >= target_epochs:
            logger.info("=" * 70)
            logger.info("=== %s: checkpoint at target (%d/%d) -- finalizing without training ===",
                        label, start_epoch, target_epochs)
            logger.info("=" * 70)
        else:
            logger.info("=" * 70)
            logger.info("=== %s: resuming epoch %d -> target %d ===", label, start_epoch, target_epochs)
            logger.info("=" * 70)
        logger.info("[%s] Loaded: epoch %d, step %d, history keys=%s",
                    label, start_epoch, global_step, list(history.keys()))
    else:
        start_epoch = 0
        global_step = 0
        history = {"epoch": []}
        logger.info("=" * 70)
        logger.info("=== %s: NO checkpoint at %s -- training FRESH from epoch 0 -> target %d ===",
                    label, ckpt_path, target_epochs)
        logger.info("=" * 70)

    def save_checkpoint(epoch, global_step, history):
        tmp_path = ckpt_path + ".tmp"
        state = {key: models[key].state_dict() for key in state_dict_keys}
        state["optimizer"] = models["optimizer"].state_dict()
        state["epoch"] = epoch
        state["global_step"] = global_step
        state["history"] = history
        torch.save(state, tmp_path)
        os.replace(tmp_path, ckpt_path)

    np.random.seed(RANDOM_SEED)
    random.seed(RANDOM_SEED)

    # --- Mixed precision: fp16 + GradScaler (Turing-friendly; bf16 is NOT faster on RTX 8000) ---
    scaler = GradScaler("cuda", enabled=(device == "cuda"))

    training_start_time = time.time()
    stop_early = False

    try:
        for epoch in range(start_epoch, target_epochs):
            epoch_losses = []
            batches = list(make_batches(size_records, batch_size))
            n_steps = len(batches)

            for step in range(n_steps):
                models["optimizer"].zero_grad()
                with autocast("cuda", dtype=torch.float16, enabled=(device == "cuda")):
                    loss = loss_fn(batches[step], models)
                scaler.scale(loss).backward()
                scaler.step(models["optimizer"])
                scaler.update()
                global_step += 1
                epoch_losses.append(loss.item())

                elapsed_hours = (time.time() - training_start_time) / 3600
                if step % LOG_EVERY_N_STEPS == 0:
                    logger.info("[%s] Epoch %d/%d, step %d/%d, loss=%.4f, elapsed=%.2fh",
                                label, epoch + 1, target_epochs, step, n_steps, loss.item(), elapsed_hours)

                if global_step % CHECKPOINT_EVERY_N_STEPS == 0:
                    save_checkpoint(epoch, global_step, history)

                if elapsed_hours > MAX_RUNTIME_HOURS_PER_SIZE:
                    logger.warning("[%s] Runtime limit reached. Saving and stopping.", label)
                    save_checkpoint(epoch, global_step, history)
                    stop_early = True
                    break
            if stop_early:
                break

            mean_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
            history["epoch"].append(epoch + 1)
            history.setdefault(loss_key, []).append(mean_loss)
            logger.info("[%s] Epoch %d done: loss=%.4f", label, epoch + 1, mean_loss)
            save_checkpoint(epoch + 1, global_step, history)

            is_gt_epoch = ((epoch + 1) % gt_every_n_epochs == 0) or ((epoch + 1) == target_epochs)
            if is_gt_epoch:
                epoch_gt_summary = eval_fn(models)
                epoch_gt_summary.insert(0, "train_size", size)
                epoch_gt_summary.insert(1, "epoch", epoch + 1)
                epoch_gt_summary.insert(2, "n_unfrozen_layers", n_unfrozen_layers)
                epoch_gt_path = os.path.join(per_epoch_gt_dir, f"epoch_{epoch + 1:02d}_gt_results.csv")
                epoch_gt_summary.to_csv(epoch_gt_path, index=False)
                logger.info("[%s] Epoch %d GT results:\n%s",
                            label, epoch + 1,
                            epoch_gt_summary.drop(columns=["train_size", "n_unfrozen_layers"]).to_string(index=False))
            else:
                logger.info("[%s] Epoch %d: GT skipped (cadence=%d)", label, epoch + 1, gt_every_n_epochs)

        pd.DataFrame(history).to_csv(os.path.join(size_dir, "training_history.csv"), index=False)
        for key in state_dict_keys:
            fname_stem = _MODEL_FILE_RENAMES.get(key, key)
            torch.save(models[key].state_dict(), os.path.join(size_dir, f"{fname_stem}.pt"))
        logger.info("[%s] Saved updated outputs.", label)

        gt_summary = eval_fn(models)
        gt_summary.insert(0, "train_size", size)
        gt_summary.insert(1, "n_unfrozen_layers", n_unfrozen_layers)
        gt_summary.to_csv(gt_results_path, index=False)
        logger.info("[%s] Final GT results:\n%s", label, gt_summary.to_string(index=False))
        return gt_summary

    except torch.cuda.OutOfMemoryError:
        logger.error("[%s] OOM. Skipping.", label)
        return None
    finally:
        try:
            del models
        except NameError:
            pass
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache(); torch.cuda.synchronize()

logger.info("train_or_resume_for_n_layers() ready (fp16 autocast + GradScaler).")

2026-10-22 10:32:19,045 : INFO : train_or_resume_for_n_layers() ready (fp16 autocast + GradScaler).


## Section 9 — Sweep over `n`

`n = 0` runs first (baseline). `n=6` **is** included here — you train your own FN-masked `n=6` for text/table.

In [12]:
results = []
for n in N_LAYERS_TO_TRY:
    logger.info("#" * 70)
    logger.info("### SWEEP: n_unfrozen_layers = %d %s###",
                n, "(frozen baseline) " if n == 0 else "")
    logger.info("#" * 70)
    r = train_or_resume_for_n_layers(
        n_unfrozen_layers=n, output_parent=OUTPUT_PARENT, size=SIZE, records_pool=texttable_records,
        batch_size=TAB_BATCH_SIZE, build_models_fn=build_models_texttable,
        loss_fn=compute_texttable_loss, eval_fn=evaluate_texttable,
        state_dict_keys=["head_text_bge_v2", "head_table_bge_v2", "bge_model"],
        loss_key="texttable_loss",
    )
    if r is not None:
        results.append(r)

logger.info("Sweep complete. %d n values produced results.", len(results))

2026-10-22 10:32:19,104 : INFO : ######################################################################
2026-10-22 10:32:19,106 : INFO : ### SWEEP: n_unfrozen_layers = 0 (frozen baseline) ###
2026-10-22 10:32:19,108 : INFO : ######################################################################
2026-10-22 10:32:19,110 : INFO : [L0] Already completed (/projects/agrifutur/sji/cross_encoder_ft/diff_unfrozen_layers/texttable_fn_masked/last_0_layers/gt_results.csv exists) -- skipping.
2026-10-22 10:32:19,138 : INFO : ######################################################################
2026-10-22 10:32:19,140 : INFO : ### SWEEP: n_unfrozen_layers = 2 ###
2026-10-22 10:32:19,141 : INFO : ######################################################################
2026-10-22 10:32:19,143 : INFO : [L2] Already completed (/projects/agrifutur/sji/cross_encoder_ft/diff_unfrozen_layers/texttable_fn_masked/last_2_layers/gt_results.csv exists) -- skipping.
2026-10-22 10:32:19,149 : INFO : ###############

In [13]:
# if interrupt 
if device == "cuda":
    torch.cuda.empty_cache(); torch.cuda.synchronize()

## Section 10 — Combined summaries

- **`summary_all_n.csv`** — final GT per `n` (`train_size, n_unfrozen_layers, source, direction, N, R@1..MRR`). All rows tagged `source=NB27B`.
- **`summary_per_epoch_all_n.csv`** — every `(n, epoch, direction)` row from `per_epoch_gt/` (epochs 2, 4, …, 20).

If you want to add NB28's texttable `size_2500` as an extra reference, set `NB28_TT_N6_REFERENCE_GT_PATH` in the config cell and re-run this section — it'll be tagged `source=NB28_ref` and appear as an extra `n=6` row alongside NB27B's own.

In [14]:
def load_nb28_reference(gt_path, expected_size=SIZE, reference_n=6):
    if not os.path.exists(gt_path):
        logger.warning("[REF n=%d] NB28 reference not found at %s", reference_n, gt_path)
        return None
    ref = pd.read_csv(gt_path)
    if "train_size" in ref.columns and not (ref["train_size"] == expected_size).all():
        logger.warning("[REF n=%d] NB28 file %s has train_size=%s, expected %d -- skipping",
                       reference_n, gt_path, ref["train_size"].unique().tolist(), expected_size)
        return None
    ref = ref.copy()
    if "train_size" not in ref.columns:
        ref.insert(0, "train_size", expected_size)
    ref.insert(1, "n_unfrozen_layers", reference_n)
    ref.insert(2, "source", "NB28_ref")
    logger.info("[REF n=%d] Loaded NB28 reference from %s (%d rows)", reference_n, gt_path, len(ref))
    return ref


def collect_summary_final(output_parent, nb28_ref_path=None):
    rows = []
    for entry in sorted(os.listdir(output_parent)):
        m = re.match(r"last_(\d+)_layers$", entry)
        if not m:
            continue
        p = os.path.join(output_parent, entry, "gt_results.csv")
        if os.path.exists(p):
            df = pd.read_csv(p)
            df["source"] = "NB27B"
            rows.append(df)
        else:
            logger.warning("No gt_results.csv at %s", p)

    if nb28_ref_path is not None:
        ref = load_nb28_reference(nb28_ref_path, expected_size=SIZE, reference_n=6)
        if ref is not None:
            rows.append(ref)

    if not rows:
        return None
    df = pd.concat(rows, ignore_index=True)
    first_cols = ["train_size", "n_unfrozen_layers", "source", "direction",
                  "N", "R@1", "R@3", "R@5", "R@10", "MRR"]
    df = df[[c for c in first_cols if c in df.columns]]
    df = (df.sort_values(["n_unfrozen_layers", "source", "direction"])
            .reset_index(drop=True))
    out = os.path.join(output_parent, "summary_all_n.csv")
    df.to_csv(out, index=False)
    logger.info("Final summary written to %s:\n%s", out, df.to_string(index=False))
    return df


def collect_summary_per_epoch(output_parent):
    rows = []
    for entry in sorted(os.listdir(output_parent)):
        m = re.match(r"last_(\d+)_layers$", entry)
        if not m:
            continue
        per_epoch_dir = os.path.join(output_parent, entry, "per_epoch_gt")
        if not os.path.isdir(per_epoch_dir):
            continue
        for f in sorted(os.listdir(per_epoch_dir)):
            if f.startswith("epoch_") and f.endswith("_gt_results.csv"):
                rows.append(pd.read_csv(os.path.join(per_epoch_dir, f)))
    if not rows:
        return None
    df = (pd.concat(rows, ignore_index=True)
            .drop_duplicates(subset=["n_unfrozen_layers", "epoch", "direction"], keep="last")
            .sort_values(["n_unfrozen_layers", "epoch", "direction"])
            .reset_index(drop=True))
    out = os.path.join(output_parent, "summary_per_epoch_all_n.csv")
    df.to_csv(out, index=False)
    logger.info("Per-epoch summary written to %s (%d rows)", out, len(df))
    return df


summary_all_n = collect_summary_final(OUTPUT_PARENT, nb28_ref_path=NB28_TT_N6_REFERENCE_GT_PATH)
summary_per_epoch_all_n = collect_summary_per_epoch(OUTPUT_PARENT)

try:
    display(summary_all_n)
except NameError:
    print(summary_all_n)

if summary_per_epoch_all_n is not None:
    last_epoch = summary_per_epoch_all_n["epoch"].max()
    compact = summary_per_epoch_all_n[summary_per_epoch_all_n["epoch"] == last_epoch]
    try:
        display(compact)
    except NameError:
        print(compact)

2026-10-22 10:32:19,371 : INFO : Final summary written to /projects/agrifutur/sji/cross_encoder_ft/diff_unfrozen_layers/texttable_fn_masked/summary_all_n.csv:
 train_size  n_unfrozen_layers source     direction   N   R@1   R@3   R@5  R@10      MRR
       2500                  0  NB27B table -> text 500 0.270 0.488 0.602 0.718 0.415567
       2500                  0  NB27B text -> table 500 0.286 0.508 0.620 0.714 0.431892
       2500                  2  NB27B table -> text 500 0.306 0.526 0.640 0.748 0.450452
       2500                  2  NB27B text -> table 500 0.282 0.532 0.638 0.736 0.439236
       2500                  4  NB27B table -> text 500 0.298 0.528 0.648 0.768 0.448303
       2500                  4  NB27B text -> table 500 0.294 0.544 0.652 0.758 0.452512
       2500                  6  NB27B table -> text 500 0.292 0.542 0.664 0.774 0.452592
       2500                  6  NB27B text -> table 500 0.302 0.544 0.648 0.754 0.454871
       2500                  8  NB27B ta

,train_size,n_unfrozen_layers,source,direction,N,R@1,R@3,R@5,R@10,MRR
0,2500,0,NB27B,table -> text,500,0.270,0.488,0.602,0.718,0.415567
1,2500,0,NB27B,text -> table,500,0.286,0.508,0.620,0.714,0.431892
2,2500,2,NB27B,table -> text,500,0.306,0.526,0.640,0.748,0.450452
3,2500,2,NB27B,text -> table,500,0.282,0.532,0.638,0.736,0.439236
4,2500,4,NB27B,table -> text,500,0.298,0.528,0.648,0.768,0.448303
5,2500,4,NB27B,text -> table,500,0.294,0.544,0.652,0.758,0.452512
6,2500,6,NB27B,table -> text,500,0.292,0.542,0.664,0.774,0.452592
7,2500,6,NB27B,text -> table,500,0.302,0.544,0.648,0.754,0.454871
8,2500,8,NB27B,table -> text,500,0.306,0.532,0.652,0.764,0.454816
9,2500,8,NB27B,text -> table,500,0.318,0.532,0.632,0.744,0.460924


,train_size,epoch,n_unfrozen_layers,direction,N,R@1,R@3,R@5,R@10,MRR
18,2500,20,0,table -> text,500,0.270,0.488,0.602,0.718,0.415567
19,2500,20,0,text -> table,500,0.286,0.508,0.620,0.714,0.431892
38,2500,20,2,table -> text,500,0.306,0.526,0.640,0.748,0.450452
39,2500,20,2,text -> table,500,0.282,0.532,0.638,0.736,0.439236
58,2500,20,4,table -> text,500,0.298,0.528,0.648,0.768,0.448303
59,2500,20,4,text -> table,500,0.294,0.544,0.652,0.758,0.452512
78,2500,20,6,table -> text,500,0.292,0.542,0.664,0.774,0.452592
79,2500,20,6,text -> table,500,0.302,0.544,0.648,0.754,0.454871
98,2500,20,8,table -> text,500,0.306,0.532,0.652,0.764,0.454816
99,2500,20,8,text -> table,500,0.318,0.532,0.632,0.744,0.460924
